In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, ttest_ind, f_oneway
import warnings
warnings.filterwarnings('ignore')

# Load data using the utility function
import sys
sys.path.append('../src')
from eda_utils import load_data

df = load_data('../data/insurance_data.csv')
print("Data shape:", df.shape)

2026-05-26 17:49:43,285 - INFO - Data loaded successfully from ../data/insurance_data.csv, shape: (10000, 21)


Data shape: (10000, 21)


Hypothesis 1: No risk differences across provinces

In [2]:
# Select two provinces
prov_a = 'Addis Ababa'
prov_b = 'Oromia'

df_prov = df[df['Province'].isin([prov_a, prov_b])].copy()

# Claim frequency
contingency = pd.crosstab(df_prov['Province'], df_prov['Claimed'])
chi2, p_freq, dof, expected = chi2_contingency(contingency)
print(f"Province comparison: {prov_a} vs {prov_b}")
print(f"Claim frequency: chi2={chi2:.3f}, p={p_freq:.4f}")

# Claim severity (only claims > 0)
severity = df_prov[df_prov['TotalClaims'] > 0]
if len(severity) > 1:
    t_stat, p_sev = ttest_ind(severity[severity['Province']==prov_a]['TotalClaims'],
                              severity[severity['Province']==prov_b]['TotalClaims'],
                              equal_var=False)
    print(f"Claim severity: t={t_stat:.3f}, p={p_sev:.4f}")
else:
    p_sev = np.nan
    print("Not enough claims for severity test.")

# Margin (profit)
df_prov['Margin'] = df_prov['TotalPremium'] - df_prov['TotalClaims']
t_stat_m, p_margin = ttest_ind(df_prov[df_prov['Province']==prov_a]['Margin'],
                               df_prov[df_prov['Province']==prov_b]['Margin'],
                               equal_var=False)
print(f"Margin: t={t_stat_m:.3f}, p={p_margin:.4f}")

Province comparison: Addis Ababa vs Oromia
Claim frequency: chi2=0.055, p=0.8139
Claim severity: t=-0.796, p=0.4265
Margin: t=0.452, p=0.6513


In [4]:
print("Unique provinces:", df['Province'].unique())
print("Unique zip codes:", df['ZipCode'].unique())
print("Unique genders:", df['Gender'].unique())

Unique provinces: <StringArray>
['Addis Ababa', 'Oromia', 'Somali', 'Tigray', 'Amhara']
Length: 5, dtype: str
Unique zip codes: [10002 10001 20001 40005 50002 30003 20005 20004 30005 30002 10004 10003
 50004 20002 30004 50001 10005 50005 40001 20003 50003 40004 30001 40002
 40003]
Unique genders: <StringArray>
['Male', 'Female']
Length: 2, dtype: str


Hypothesis 2 & 3: Zip codes (risk and margin differences)

In [5]:
# Choose two zip codes that actually exist (replace with your actual values)
zip_a = 10002   # example – change to a real value from df['ZipCode'].unique()
zip_b = 20003   # example – change to a real value

df_zip = df[df['ZipCode'].isin([zip_a, zip_b])].copy()

if len(df_zip) == 0:
    print("No data for selected zip codes. Check the unique zip codes above.")
else:
    # Claim frequency
    contingency_zip = pd.crosstab(df_zip['ZipCode'], df_zip['Claimed'])
    if contingency_zip.size == 0:
        print("Not enough data for chi-squared test.")
    else:
        chi2_z, p_freq_z, dof_z, exp_z = chi2_contingency(contingency_zip)
        print(f"Zip codes: {zip_a} vs {zip_b}")
        print(f"Claim frequency: chi2={chi2_z:.3f}, p={p_freq_z:.4f}")

    # Margin
    df_zip['Margin'] = df_zip['TotalPremium'] - df_zip['TotalClaims']
    if len(df_zip) > 1:
        t_stat_mz, p_margin_z = ttest_ind(df_zip[df_zip['ZipCode']==zip_a]['Margin'],
                                          df_zip[df_zip['ZipCode']==zip_b]['Margin'],
                                          equal_var=False)
        print(f"Margin: t={t_stat_mz:.3f}, p={p_margin_z:.4f}")
    else:
        print("Not enough data for margin t-test.")

Zip codes: 10002 vs 20003
Claim frequency: chi2=0.003, p=0.9534
Margin: t=-0.139, p=0.8894


 Hypothesis 4: Gender (male vs female)

In [6]:
# Gender groups
df_gender = df.copy()
# Claim frequency
contingency_gender = pd.crosstab(df_gender['Gender'], df_gender['Claimed'])
chi2_g, p_freq_g, dof_g, exp_g = chi2_contingency(contingency_gender)
print(f"Gender: male vs female")
print(f"Claim frequency: chi2={chi2_g:.3f}, p={p_freq_g:.4f}")

# Claim severity
severity_g = df_gender[df_gender['TotalClaims'] > 0]
if len(severity_g) > 1:
    t_stat_g, p_sev_g = ttest_ind(severity_g[severity_g['Gender']=='Male']['TotalClaims'],
                                  severity_g[severity_g['Gender']=='Female']['TotalClaims'],
                                  equal_var=False)
    print(f"Claim severity: t={t_stat_g:.3f}, p={p_sev_g:.4f}")
else:
    p_sev_g = np.nan
    print("Not enough claims for severity test.")

# Margin
df_gender['Margin'] = df_gender['TotalPremium'] - df_gender['TotalClaims']
t_stat_mg, p_margin_g = ttest_ind(df_gender[df_gender['Gender']=='Male']['Margin'],
                                  df_gender[df_gender['Gender']=='Female']['Margin'],
                                  equal_var=False)
print(f"Margin: t={t_stat_mg:.3f}, p={p_margin_g:.4f}")

Gender: male vs female
Claim frequency: chi2=0.002, p=0.9638
Claim severity: t=0.005, p=0.9964
Margin: t=0.019, p=0.9847


| Hypothesis | Comparison | KPI | Test | p‑value | Decision (α=0.05) | Business Interpretation |
|------------|------------|-----|------|---------|-------------------|-------------------------|
| Provinces | Addis Ababa vs Oromia | Claim Frequency | Chi‑squared | 0.8139 | Fail to reject H₀ | No significant difference in claim frequency between these two provinces. |
| Provinces | Addis Ababa vs Oromia | Claim Severity | t‑test | 0.4265 | Fail to reject H₀ | No significant difference in claim severity. |
| Provinces | Addis Ababa vs Oromia | Margin | t‑test | 0.6513 | Fail to reject H₀ | No significant difference in profitability. |
| Zip codes | 10002 vs 20003 | Claim Frequency | Chi‑squared | 0.9534 | Fail to reject H₀ | No significant difference in claim frequency between these zip codes. |
| Zip codes | 10002 vs 20003 | Margin | t‑test | 0.8894 | Fail to reject H₀ | No significant difference in profitability. |
| Gender | Male vs Female | Claim Frequency | Chi‑squared | 0.9638 | Fail to reject H₀ | No significant risk difference by gender. |
| Gender | Male vs Female | Claim Severity | t‑test | 0.9964 | Fail to reject H₀ | Claim severity does not differ by gender. |
| Gender | Male vs Female | Margin | t‑test | 0.9847 | Fail to reject H₀ | Profitability does not differ by gender. |

## Business Recommendations

Based on the hypothesis tests (all p‑values > 0.05), we do **not** have statistical evidence that the selected provinces, zip codes, or gender drive risk differences. This suggests that:

- **Pricing should not be adjusted solely based on province (at least for Addis Ababa vs Oromia) or gender.**  
- **Zip code alone does not explain differences in claim frequency or margin for the tested pair.**

**However**, the lack of statistical significance could be due to:
- Small sample sizes for certain groups.
- Other unobserved confounding factors (vehicle type, age, etc.).

**Next steps:**
1. Test additional province pairs (e.g., Somali vs Tigray) that may show differences.
2. Use multivariate analysis (e.g., logistic regression for claim probability) to control for other variables.
3. Continue with predictive modeling (Task 4) to capture non‑linear interactions.